Step 1: Set up the environment and import libraries.

In [ ]:
# !pip install albumentations scikit-learn

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Step 2: Process Labels and Split Train/Test Files

In [ ]:
# PATH TO DATA (Update to match your Kaggle path or dataset)
DATA_DIR = 'nih-chest-x-ray-14-224x224-resized'
IMAGE_DIR = os.path.join(DATA_DIR, 'images-224/images-224')
CSV_PATH = os.path.join(DATA_DIR, 'Data_Entry_2017.csv')
TRAIN_TXT = os.path.join(DATA_DIR, 'train_val_list_NIH.txt')
TEST_TXT = os.path.join(DATA_DIR, 'test_list_NIH.txt')

# 1. List of 14 diseases (Ignore "No Finding", because if these 14 diseases are 0, the default is No Finding)
DISEASES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema',
    'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

# 2. Read CSV
df = pd.read_csv(CSV_PATH)
df = df[['Image Index', 'Finding Labels']] # Take only the two necessary columns.

#3. Create columns 0/1 for 14 diseases
for disease in DISEASES:
    df[disease] = df['Finding Labels'].apply(lambda x: 1.0 if disease in x else 0.0)

# 4. Read the list of Train/Test
with open(TRAIN_TXT, 'r') as f:
    train_images = [line.strip() for line in f.readlines()]

with open(TEST_TXT, 'r') as f:
    test_images = [line.strip() for line in f.readlines()]

#5. Filtering DataFrames
train_df = df[df['Image Index'].isin(train_images)].reset_index(drop=True)
val_df = df[df['Image Index'].isin(test_images)].reset_index(drop=True)

print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)}")

Step 3: Define PyTorch Dataset & Dataloader

In [ ]:
class NIHChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
        self.labels = self.dataframe[DISEASES].values # Get the label matrix
        self.image_names = self.dataframe['Image Index'].values

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.image_dir, img_name)

        # Reads RGB images (because pre-trained ImageNet requires 3 color channels)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return image, label

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(
        translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)}, # The corresponding shift_limit=0.05
        scale=(0.95, 1.05),                                         # The corresponding scale_limit=0.05
        rotate=(-10, 10),                                           # The corresponding rotate_limit=10
        p=0.5
    ),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

BATCH_SIZE = 64 # Since 224x224 images are quite small, you can use a larger batch_size.

train_dataset = NIHChestXrayDataset(train_df, IMAGE_DIR, transform=train_transform)
val_dataset = NIHChestXrayDataset(val_df, IMAGE_DIR, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

Step 4: Initialize the DenseNet121 Model with ImageNet Weights

In [ ]:
class NIH_DenseNet121(nn.Module):
    def __init__(self, num_classes=14):
        super().__init__()
        # Load a pre-trained model from PyTorch's ImageNet.
        self.model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

        # Replace the last classifier class (1000 classes -> 14 classes)
        num_ftrs = self.model.classifier.in_features
        self.model.classifier = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.model(x)

model = NIH_DenseNet121(num_classes=14)
model = model.to(device)

Step 5: Training Loop

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=1)

EPOCHS = 10
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    # ================= TRAIN =================
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for images, targets in pbar:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})

    avg_train_loss = train_loss / len(train_loader)

    # ================= VALIDATION =================
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        pbar_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")
        for images, targets in pbar_val:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"\nEpoch {epoch+1} Summary: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}\n")

    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr}")

    # Save the .pth file if the loss decreases.
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_nih_densenet121.pth')
        print(f"The best model has been saved at Epoch. {epoch+1}!")